# NES-VMC（Natural Excited State Variational Monte Carlo）算法实现

## 关于 Model 输出的重要说明

**NetKet 约定**：在 NetKet 中，神经网络 model 的输出默认是 **log 取值**，即输出 $\log \psi(x)$ 而不是 $\psi(x)$。

这意味着：
- 局部能量计算：$\frac{\psi(\eta)}{\psi(\sigma)} = \exp(\log \psi(\eta) - \log \psi(\sigma))$
- 采样：基于 $|\psi|^2 = \exp(2\text{Re}(\log \psi))$

在 NES-VMC 中，我们需要：
- 行列式计算：$\det(\Psi)$ 其中 $\Psi_{ij} = \psi_j(x^i)$
- 由于 $\det(\exp A) = \exp(\text{Tr}(A))$，我们可以使用对数形式

$$\nabla_{\theta} \mathcal{L} = 2 \mathbb{E}_{\mathbf{x} \sim |\Psi|^2} \left[ \mathrm{Tr}\left( E_L(\mathbf{x}) - \bar{E}_L \right) \nabla_{\theta} \log \Psi(\mathbf{x})^* \right]$$

## 1. 环境配置与 FCI 基准

In [ ]:
# ===================== 环境配置 =====================
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from functools import partial
from jax import flatten_util
import matplotlib.pyplot as plt
from tqdm import tqdm
from jax import vmap,jit,grad
from NES_VMC import create_machine,hi_ext,hi,sampler,SingleStateAnsatz,NESTotalAnsatz,ha,compute_loss_and_grad,E_fcis



In [ ]:
class NESTotalAnsatz(nnx.Module):
    """
    NES-VMC 总波函数 Ansatz
    支持自动处理：
      - 单样本：输入 (K, n_spin_orbitals)
      - 批量样本：输入 (batch_size, K, n_spin_orbitals)
    输出：
      log_psi_total : 标量 或 (batch_size,)
      log_M_matrix  : (K,K) 或 (batch_size,K,K)
    """
    def __init__(self, n_spin_orbitals: int, n_states: int = 2, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.K = n_states
        self.n_spin = n_spin_orbitals

        # 初始化 K 个独立单态波函数
        self.single_ansatz_list = [
            SingleStateAnsatz(n_spin_orbitals, hidden_dim, rngs=rngs)
            for _ in range(self.K)
        ]

    def __call__(self, x: jax.Array):
        # ----------------------
        # 单样本处理函数 (核心)
        # x_single: (K, n_spin)
        # ----------------------
        def _forward_single(x_single):
            # 构造 K×K 矩阵 M: M[i,j] = ψ_j(x_i)
            #print(f'x_single.shape={x_single.shape}')
            M = []
            for i in range(self.K):
                row = []
                for j in range(self.K):
                    val = self.single_ansatz_list[j](x_single[i])
                    row.append(val)
                M.append(jnp.stack(row))
            M = jnp.stack(M)  # (K, K)
            log_det = jnp.linalg.det(M)
            return log_det, M

        # ----------------------
        # 自动判断：单条 / 批量
        # ----------------------
        if x.shape[-1] == self.n_spin:
            
            #print('A')
            log_psi, log_M = _forward_single(x)
        else:
            #print('B')
            x = x.reshape(-1, K, self.n_spin)
            #print(f'转换后x.shape={x.shape}')
            log_psi, log_M = jax.vmap(_forward_single)(x)
        return log_psi, log_M
def create_machine(model: NESTotalAnsatz):
    """将 Flax NNX 模型包装为 NetKet 风格的 machine 函数"""
    graphdef, state = nnx.split(model)

    @jax.jit
    def machine(params, sigma):
        #print(f'x.shape: {sigma.shape}  ')
        m = nnx.merge(graphdef, params)
        log_psi_total,log_M_matrix = m(sigma)
        return log_psi_total

    return machine, graphdef, state

In [ ]:
def compute_qgt(machine, params, sigma, diag_shift=0.1):
    """
    计算量子几何张量（QGT）/ F 矩阵
    
    QGT 定义：
    S_ij = ⟨∂_i log ψ* ∂_j log ψ⟩ - ⟨∂_i log ψ*⟩⟨∂_j log ψ⟩
    
    这就是 NetKet SR 的核心
    
    参数：
    - machine: 波函数机器
    - params: 网络参数
    - sigma: 样本 (n_samples, n_orbitals)
    - diag_shift: 对角线正则化参数 λ
    
    返回：
    - qgt_reg: 正则化后的 QGT 矩阵 (n_params, n_params)
    - unravel_fn: 用于将展平的向量恢复为 PyTree 结构的函数
    """
    sigma = sigma.reshape(-1,2,4)
    n_samples = sigma.shape[0]
    
    # 步骤 1: 计算每个样本的 ∇log ψ
    def log_psi_single(p, s):
        return machine(p, s)
    
    def compute_grad_for_sample(s):
        return jax.grad(lambda p: log_psi_single(p, s), holomorphic=True)(params)
    
    # grad_matrix 是 PyTree，每个元素形状为 (n_samples, ...)
    grad_matrix = jax.vmap(compute_grad_for_sample)(sigma)
    
    # 步骤 2: 将 PyTree 展平为矩阵 (n_samples, n_params)
    grad_flat, unravel_fn = flatten_util.ravel_pytree(grad_matrix)
    grad_flat = grad_flat.reshape(n_samples, -1)
    
    # 步骤 3: 中心化（减去均值）
    # 这对应 QGT 定义中的第二项：- ⟨∂_i log ψ*⟩⟨∂_j log ψ⟩
    grad_mean = jnp.mean(grad_flat, axis=0, keepdims=True)  # (1, n_params)
    grad_centered = grad_flat - grad_mean  # (n_samples, n_params)
    
    # 步骤 4: 计算 QGT = (1/N) * Σ ∇log ψ* ∇log ψ^T
    # 注意：对于复数，需要使用共轭
    qgt = (1.0 / n_samples) * jnp.conj(grad_centered).T @ grad_centered
    
    # 步骤 5: 添加正则化
    qgt_reg = qgt + diag_shift * jnp.eye(qgt.shape[0])
    
    return qgt_reg, unravel_fn

In [ ]:
K=2
lr=0.01
n_iter = 300
# 初始化模型
rngs = nnx.Rngs(21)
total_ansatz = NESTotalAnsatz(4, K, 8, rngs=rngs)
machine, graphdef, params = create_machine(total_ansatz)

# 初始化采样器状态
sampler_state = sampler.init_state(machine, params,seed=12)

# 初始化优化器
optimizer = optax.sgd(lr)
opt_state = optimizer.init(params)

# 训练循环
E_L_history = []
loss_history = []

print(f'标准答案是基态能量{E_fcis[0]:.8f}｜第一激发态能量{E_fcis[1]:.8f}')
for step in range(n_iter):
    # 1. 采样
    samples, sampler_state = sampler.sample(
        machine, params, state=sampler_state, chain_length=20
    )    
    # 2. 计算损失和梯度
    loss, grads, E_L_avg = compute_loss_and_grad(
        graphdef, params, samples, ha, K
    )
    
    grad_flat , grad_unravel_fn = flatten_util.ravel_pytree(grads)
    qgt_reg,qgt_unravel_fn = compute_qgt(machine, params, samples,0.001) 
    # qgt_flat , qgt_unravel_fn = flatten_util.ravel_pytree(qgt_reg)
    
    natural_grad = jnp.linalg.solve(qgt_reg, grad_flat)
    natural_grad = grad_unravel_fn(natural_grad)
    grad = natural_grad
    # 3. 参数更新
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    # 4. 记录和打印
    loss_history.append(loss)
    E_L_history.append(E_L_avg)
    
    if step % 10 == 0:
        # 对角化平均局域能量矩阵获取激发态能量
        eig_vals = jnp.linalg.eigvalsh(E_L_avg)
        print(f"Step {step:4d} | Loss: {loss:.8f} | Energies: {eig_vals}")


In [ ]:
import jax
import jax.numpy as jnp
import optax
from flax import nnx
from functools import partial
from jax import vmap, grad

# ==============================================================================
# ✅ 1. 【严格协方差版】QGT 计算（VMC / NES-VMC 标准定义）
# ==============================================================================
@partial(jax.jit, static_argnames=("model_graphdef",))
def compute_QGT(model_graphdef, params, sigma):
    """
    标准 QGT = Cov( ∇logΨ, ∇logΨ† )
    完全符合你说的：QGT = 梯度协方差矩阵
    """
    def log_psi(p, x):
        model = nnx.merge(model_graphdef, p)
        log_p, _ = model(x)
        return jnp.real(log_p)  # log|Ψ|

    # 计算批量梯度 ∇logΨ
    grad_log = grad(log_psi, argnums=0)
    batch_g = vmap(grad_log, (None, 0))(params, sigma)  # (B, ...)

    # 均值 E[g]
    mean_g = jax.tree.map(lambda g: jnp.nanmean(g, axis=0), batch_g)

    # 协方差 E[gg†] - E[g]E[g]†
    def qgt_cov(g, mg):
        eg = jnp.nanmean(g * jnp.conj(g), axis=0)
        return eg - mg * jnp.conj(mg)

    S = jax.tree.map(qgt_cov, batch_g, mean_g)
    return S

# ==============================================================================
# ✅ 2. 自然梯度：grad_ng = S⁻¹ · grad (带小正则防止除零)
# ==============================================================================
@jax.jit
def apply_natural_gradient(grads, S, eps=1e-4):
    return jax.tree.map(lambda g, s: g / (s + eps), grads, S)

# ==============================================================================
# ✅ 3. 你的训练主代码（已加入 QGT + 自然梯度）
# ==============================================================================

# 超参数
K = 2
lr = 0.1
n_iter = 100

# 初始化模型
rngs = nnx.Rngs(21)
total_ansatz = NESTotalAnsatz(4, K, 12, rngs=rngs)
machine, graphdef, params = create_machine(total_ansatz)

# 采样器
sampler_state = sampler.init_state(machine, params,seed=12)

# 优化器
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

# 日志
E_L_history = []
loss_history = []

# 训练循环
for step in range(n_iter):
    # 1. 采样
    samples, sampler_state = sampler.sample(
        machine, params, state=sampler_state, chain_length=20
    )
    samples = samples.reshape(-1, K, 4)  # (B, K, n_spin)

    # 2. 计算损失、标准梯度、平均局域能量矩阵
    loss, grads, E_L_avg = compute_loss_and_grad(
        graphdef, params, samples, ha, K
    )

    # ========================
    # 🔥 核心新增：QGT + 自然梯度
    # ========================
    S = compute_QGT(graphdef, params, samples)
    grads_natural = apply_natural_gradient(grads, S)

    # 3. 使用自然梯度更新参数
    updates, opt_state = optimizer.update(grads_natural, opt_state, params)
    params = optax.apply_updates(params, updates)

    # 4. 日志
    loss_history.append(loss)
    E_L_history.append(E_L_avg)

    # 5. 输出能量（对角化 E_L_mean）
    if step % 10 == 0:
        eig_vals = jnp.linalg.eigvalsh(E_L_avg)
        print(f"Step {step:4d} | Loss: {loss:.8f} | Energies: {eig_vals}")

## 4. 哈密顿量作用计算

In [ ]:
# ===================== 哈密顿量作用计算 =====================
def Ham_psi(ha, model, x):
    """计算 Hψ(x)"""
    x_primes, mels = ha.get_conn(x)
    psi_values = jax.vmap(model)(x_primes)
    H_psi_x = jnp.sum(mels * psi_values)
    return H_psi_x


def Ham_Psi(ha, total_ansatz, x):
    """计算扩展哈密顿量作用在总 Ansatz 上的矩阵"""
    k = total_ansatz.n_states
    if x.shape[0] != k:
        raise ValueError(f"Input array must have shape ({k},) but got shape {x.shape}")

    H_psi_x_i = []
    for i in range(k):
        tmp = []
        for j in range(k):
            ele = Ham_psi(ha, model=total_ansatz.single_ansatz_list[j], x=x[i])
            tmp.append(ele)
        H_psi_x_i.append(tmp)

    HPsi = jnp.array(H_psi_x_i).reshape(k, k)
    return HPsi

print("哈密顿量作用计算函数定义完成!")

## 5. 局域能量矩阵

In [ ]:
# ===================== 局域能量矩阵计算 =====================
def compute_local_energy_matrix_safe(ha, total_ansatz, x_batch, eps=1e-8):
    """安全计算局域能量矩阵，带数值稳定性保护"""
    psi_total, M_matrix = total_ansatz(x_batch)
    det_M = jnp.linalg.det(M_matrix)
    abs_det = jnp.abs(det_M)

    M_reg = M_matrix
    if abs_det < eps:
        M_reg = M_matrix + eps * jnp.eye(M_matrix.shape[0])

    H_Psi = Ham_Psi(ha, total_ansatz, x_batch)
    E_L = jnp.linalg.solve(M_reg, H_Psi.T).T
    return E_L, psi_total, det_M

print("局域能量矩阵计算函数定义完成!")

## 6. 损失函数与梯度

In [ ]:
# ===================== NES 损失函数与梯度计算 =====================

def compute_local_energy_matrix_from_params_safe(ha, graphdef, params, x_batch, n_states, eps=1e-8):
    """从参数计算局域能量矩阵，带数值稳定性保护"""
    merged_model = nnx.merge(graphdef, params)

    K = n_states
    M = []
    for i in range(K):
        for j in range(K):
            psi_i_xj = merged_model.single_ansatz_list[j](x_batch[i])
            M.append(psi_i_xj)

    M_matrix = jnp.stack(M).reshape(K, K)
    det_M = jnp.linalg.det(M_matrix)
    abs_det = jnp.abs(det_M)

    M_reg = M_matrix
    if abs_det < eps:
        M_reg = M_matrix + eps * jnp.eye(K)

    H_Psi = []
    for i in range(K):
        row = []
        for j in range(K):
            x_primes, mels = ha.get_conn(x_batch[i])
            psi_values = jax.vmap(lambda x: merged_model.single_ansatz_list[j](x))(x_primes)
            H_psi = jnp.sum(mels * psi_values)
            row.append(H_psi)
        H_Psi.append(row)

    H_Psi_matrix = jnp.array(H_Psi).reshape(K, K)
    E_L = jnp.linalg.solve(M_reg, H_Psi_matrix.T).T
    psi_total = jnp.linalg.det(M_matrix)

    return E_L, psi_total, det_M


def sample_nes_batches(sampler, machine_fn, params, sampler_state, n_samples, K, n_spin_orbitals):
    """从扩展希尔伯特空间采样 NES 批次"""
    samples_list = []

    for _ in range(n_samples):
        sample, sampler_state = sampler.sample(machine_fn, params, state=sampler_state)
        samples_list.append(sample.reshape(-1, n_spin_orbitals))

    all_samples = jnp.stack(samples_list).reshape(-1, n_spin_orbitals)

    x_batches = []
    for i in range(n_samples // K):
        indices = jnp.arange(i * K, (i + 1) * K)
        batch = all_samples[indices]
        x_batches.append(batch)

    return jnp.stack(x_batches), sampler_state


def compute_nes_loss_and_grad(ha, graphdef, params, sigma_batches, n_states, eps=1e-8):
    """计算 NES-VMC 的损失函数和梯度，带数值稳定性保护"""
    E_L_list = []
    valid_count = 0

    for i in range(sigma_batches.shape[0]):
        E_L, _, det_M = compute_local_energy_matrix_from_params_safe(
            ha, graphdef, params, sigma_batches[i], n_states, eps
        )

        abs_det = jnp.abs(det_M)
        is_valid = jnp.isfinite(E_L).all() and abs_det > eps

        if is_valid:
            E_L_list.append(E_L)
            valid_count += 1

    if valid_count == 0:
        print("Warning: No valid samples!")
        return jnp.array(np.nan), jax.tree.map(lambda x: jnp.zeros_like(x), params), jnp.array([])

    E_L_batch = jnp.stack(E_L_list)
    loss = jnp.trace(jnp.mean(E_L_batch, axis=0))

    grads = []
    for i in range(len(E_L_list)):
        def loss_per_sample(p):
            E_L_s, _, _ = compute_local_energy_matrix_from_params_safe(
                ha, graphdef, p, sigma_batches[i], n_states, eps
            )
            return jnp.trace(E_L_s)

        grad_i = jax.grad(loss_per_sample, holomorphic=True)(params)
        grads.append(grad_i)

    grad = jax.tree.map(
        lambda *x: jnp.mean(jnp.stack(x), axis=0),
        *grads
    )

    return loss, grad, E_L_batch


def extract_eigenvalues(E_L_matrices):
    """从局域能量矩阵提取本征值（激发态能量）"""
    if len(E_L_matrices) == 0:
        return jnp.array([jnp.nan, jnp.nan]), None, None

    E_L_mean = jnp.mean(E_L_matrices, axis=0)

    if not jnp.isfinite(E_L_mean).all():
        return jnp.array([jnp.nan, jnp.nan]), None, E_L_mean

    eigenvalues, eigenvectors = jnp.linalg.eigh(E_L_mean)
    return eigenvalues, eigenvectors, E_L_mean

print("损失函数与梯度计算函数定义完成!")

## 7. 训练循环

In [ ]:
# ===================== 训练参数设置 =====================
K = 2  # 态数量
n_spin_orbitals = 4
hidden_dim = 16
learning_rate = 0.01
n_iterations = 100
n_samples = 1008

print(f"训练参数:")
print(f"  - K (态数量): {K}")
print(f"  - n_spin_orbitals: {n_spin_orbitals}")
print(f"  - hidden_dim: {hidden_dim}")
print(f"  - 学习率: {learning_rate}")
print(f"  - 迭代次数: {n_iterations}")
print(f"  - 样本数: {n_samples}")

In [ ]:
# ===================== 初始化模型和优化器 =====================
total_ansatz = NESTotalAnsatz(
    n_spin_orbitals=n_spin_orbitals,
    n_states=K,
    hidden_dim=hidden_dim,
    rngs=nnx.Rngs(42)
)

graphdef, params = nnx.split(total_ansatz)

# 创建用于采样的 machine 函数（单态）
single_machine = total_ansatz.single_ansatz_list[0]
single_graphdef, single_state = nnx.split(single_machine)

@jax.jit
def sampling_machine(params, sigma):
    model = nnx.merge(single_graphdef, params)
    return model(sigma)

sampler_state = sampler.init_state(sampling_machine, single_state, seed=42)

optimizer = optax.sgd(learning_rate=learning_rate)
opt_state = optimizer.init(params)

print("模型和优化器初始化完成!")

In [ ]:
# ===================== 训练循环 =====================
print("\n" + "="*60)
print("开始 NES-VMC 训练")
print("="*60)

history = {
    'step': [],
    'loss': [],
    'eigenvalues': [],
    'error_0': [],
    'error_1': []
}

for step in tqdm(range(n_iterations)):
    # 采样
    sigma_batches, sampler_state = sample_nes_batches(
        sampler, sampling_machine, single_state, sampler_state,
        n_samples, K, n_spin_orbitals
    )

    # 计算损失函数和梯度
    loss, grad, E_L_batch = compute_nes_loss_and_grad(
        ha, graphdef, params, sigma_batches, K
    )

    # 提取本征值
    eigenvalues, eigenvectors, E_L_mean = extract_eigenvalues(E_L_batch)

    # 参数更新
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)

    if step % 1 == 0:
        history['step'].append(step)
        history['loss'].append(float(loss.real) if np.isfinite(loss.real) else np.nan)
        history['eigenvalues'].append([float(e) if np.isfinite(e) else np.nan for e in eigenvalues])
        history['error_0'].append(float(abs(eigenvalues[0] - E_fcis[0])) if np.isfinite(eigenvalues[0]) else np.nan)
        history['error_1'].append(float(abs(eigenvalues[1] - E_fcis[1])) if np.isfinite(eigenvalues[1]) else np.nan)

        print(f"\nStep {step:3d} | Loss: {loss.real if np.isfinite(loss.real) else 'NaN':.8f}")
        if np.isfinite(eigenvalues[0]):
            print(f"  本征值: E_0 = {eigenvalues[0]:.8f} (FCI: {E_fcis[0]:.8f}, Error: {abs(eigenvalues[0] - E_fcis[0]):.6f})")
            print(f"          E_1 = {eigenvalues[1]:.8f} (FCI: {E_fcis[1]:.8f}, Error: {abs(eigenvalues[1] - E_fcis[1]):.6f})")
        else:
            print("  本征值: NaN")

print("\n" + "="*60)
print("训练完成!")
print("="*60)

## 8. 结果可视化

In [ ]:
# ===================== 结果可视化 =====================
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

valid_steps = [i for i, v in enumerate(history['loss']) if not np.isnan(v)]
if len(valid_steps) > 0:
    valid_loss = [v for v in history['loss'] if not np.isnan(v)]
    valid_steps_arr = history['step'][:len(valid_loss)]
    
    axes[0, 0].plot(valid_steps_arr, valid_loss, 'b-', linewidth=2)
    axes[0, 0].set_xlabel('Iteration Step')
    axes[0, 0].set_ylabel('Loss (Tr(E_L))')
    axes[0, 0].set_title('NES-VMC Loss Convergence')
    axes[0, 0].grid(True, alpha=0.3)

    eigenvalues_array = np.array(history['eigenvalues'][:len(valid_loss)])
    for i in range(K):
        valid_evs = eigenvalues_array[:, i]
        valid_mask = ~np.isnan(valid_evs)
        if np.any(valid_mask):
            axes[0, 1].plot(valid_steps_arr[valid_mask], valid_evs[valid_mask],
                            label=f'E_{i}', linewidth=2)
    axes[0, 1].axhline(y=E_fcis[0], color='r', linestyle='--',
                       label=f'FCI E0: {E_fcis[0]:.4f}')
    axes[0, 1].axhline(y=E_fcis[1], color='g', linestyle='--',
                       label=f'FCI E1: {E_fcis[1]:.4f}')
    axes[0, 1].set_xlabel('Iteration Step')
    axes[0, 1].set_ylabel('Energy (Ha)')
    axes[0, 1].set_title('Eigenvalue Convergence')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    valid_error_0 = [v for v in history['error_0'] if not np.isnan(v)]
    valid_error_1 = [v for v in history['error_1'] if not np.isnan(v)]
    if len(valid_error_0) > 0:
        axes[1, 0].semilogy(valid_steps_arr[:len(valid_error_0)], valid_error_0, 'b-',
                         linewidth=2, label='Error E_0')
    if len(valid_error_1) > 0:
        axes[1, 0].semilogy(valid_steps_arr[:len(valid_error_1)], valid_error_1, 'r-',
                         linewidth=2, label='Error E_1')
    axes[1, 0].set_xlabel('Iteration Step')
    axes[1, 0].set_ylabel('Absolute Error (Ha)')
    axes[1, 0].set_title('Error vs FCI Reference')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3, which='both')

    if len(valid_loss) > 0 and not np.isnan(eigenvalues_array[-1, 0]):
        axes[1, 1].bar(['FCI E0', 'NES E0', 'FCI E1', 'NES E1'],
                       [E_fcis[0], eigenvalues_array[-1, 0],
                        E_fcis[1], eigenvalues_array[-1, 1]],
                       color=['red', 'blue', 'red', 'blue'],
                       alpha=0.7)
    else:
        axes[1, 1].text(0.5, 0.5, 'No valid results', ha='center', va='center')
    axes[1, 1].set_ylabel('Energy (Ha)')
    axes[1, 1].set_title('Final Energy Comparison')
    axes[1, 1].grid(True, alpha=0.3, axis='y')
else:
    axes[0, 0].text(0.5, 0.5, 'No valid training results', ha='center', va='center')
    axes[0, 1].text(0.5, 0.5, 'No valid results', ha='center', va='center')
    axes[1, 0].text(0.5, 0.5, 'No valid results', ha='center', va='center')
    axes[1, 1].text(0.5, 0.5, 'No valid results', ha='center', va='center')

plt.tight_layout()
plt.savefig('nes_vmc_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n结果图表已保存为 'nes_vmc_results.png'")